# LLM API Mastery: SarvamAI, OpenAI, and OpenRouter
## A Practical Notebook for AI Application Development

This notebook covers:
- SarvamAI API integration and token usage
- OpenAI GPT-4o Mini with full parameter control
- OpenRouter for multi-model access
- BM25 retrieval combined with LLM calls
- Multi-turn conversation with context management
- Temperature, Top-P, Top-K, Penalty parameters with real-world scenarios
- Prompt engineering techniques

---


## Section 0: Install Required Libraries

In [1]:
# Install all required libraries
# Run this cell first before anything else

!pip install sarvamai openai rank_bm25 requests -q

print("All libraries installed successfully.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.7/265.7 kB 6.0 MB/s eta 0:00:00
All libraries installed successfully.


---
## Section 1: SarvamAI API - Basics to Advanced

SarvamAI is an Indian LLM provider with strong support for Indian languages.
We will start from a basic call and build up to full parameter control.


### 1.1 Basic API Call

In [3]:
from sarvamai import SarvamAI

# Replace with your actual API key
SARVAM_API_KEY = ""

client = SarvamAI(api_subscription_key=SARVAM_API_KEY)

response = client.chat.completions(
    model="sarvam-105b",
    messages=[
        {"role": "user", "content": "Explain the economic impact of GST implementation in India."}
    ],
    temperature=0.5,
    top_p=1,
    max_tokens=3000,
)

print(response.choices[0].message.content)


Of course. The implementation of the Goods and Services Tax (GST) on July 1, 2017, was one of the most significant tax reforms in India's history. It aimed to replace a complex web of multiple central and state taxes with a single, unified tax. The economic impact has been multifaceted, with notable successes alongside persistent challenges.

Here is a detailed explanation of the economic impact of GST implementation in India.

---

### **1. The Rationale: Expected Economic Benefits**

Before examining the actual impact, it's essential to understand the goals of GST:

*   **To Eliminate the Cascading Effect of Tax:** The previous system had a "tax-on-tax" structure, where tax was levied on tax at each stage of the supply chain. GST aimed to solve this through a seamless **Input Tax Credit (ITC)** mechanism, where a business could claim credit for the tax paid on its inputs.
*   **To Create a "One Nation, One Tax" Market:** The goal was to remove interstate tax barriers, making it easie

### 1.2 Print Everything From the Response Object

In [5]:
import json

# Method 1: Print the full response object
print("--- Full Response Object ---")
print(response)

print()

# Method 2: Pretty-print using __dict__
print("--- Pretty Printed Response ---")
try:
    print(json.dumps(response.__dict__, indent=2, default=str))
except Exception as e:
    print("Direct dict not available. Using model_dump if available.")

# Method 3: Print all important fields explicitly
print()
print("--- Explicit Fields ---")
print("Model Used:", response.model if hasattr(response, 'model') else "N/A")
print("Number of Choices:", len(response.choices))

for i, choice in enumerate(response.choices):
    print(f"Choice {i}:")
    print("  Role:", choice.message.role)
    print("  Content:", choice.message.content)
    print("  Finish Reason:", choice.finish_reason if hasattr(choice, 'finish_reason') else "N/A")

print()
print("--- Usage Info ---")
if hasattr(response, 'usage'):
    print("Usage:", response.usage)

# Method 4: See all available attributes
print()
print("--- All Attributes ---")
print(dir(response))


--- Full Response Object ---
id='20260415_741d52c4-7f40-4af8-b441-7f0a02f44dc3' choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionResponseMessage(content='Of course. The implementation of the Goods and Services Tax (GST) on July 1, 2017, was one of the most significant tax reforms in India\'s history. It aimed to replace a complex web of multiple central and state taxes with a single, unified tax. The economic impact has been multifaceted, with notable successes alongside persistent challenges.\n\nHere is a detailed explanation of the economic impact of GST implementation in India.\n\n---\n\n### **1. The Rationale: Expected Economic Benefits**\n\nBefore examining the actual impact, it\'s essential to understand the goals of GST:\n\n*   **To Eliminate the Cascading Effect of Tax:** The previous system had a "tax-on-tax" structure, where tax was levied on tax at each stage of the supply chain. GST aimed to solve this through a seamless **Input Tax Credit

### 1.3 Token Usage Calculator Function

In [6]:
def get_token_usage(response):
    """
    Extract token usage from a SarvamAI or OpenAI response.
    Handles both object-style and string-style usage fields.

    Returns a dict with prompt_tokens, completion_tokens, total_tokens.
    """
    usage_data = response.usage

    # Case 1: Usage is already a structured object (standard OpenAI style)
    if hasattr(usage_data, "prompt_tokens"):
        return {
            "prompt_tokens": usage_data.prompt_tokens,
            "completion_tokens": usage_data.completion_tokens,
            "total_tokens": usage_data.total_tokens
        }

    # Case 2: Usage is returned as a string (SarvamAI sometimes does this)
    # Format: "completion_tokens=3210 prompt_tokens=20 total_tokens=3230"
    elif isinstance(usage_data, str):
        usage_dict = {}
        parts = usage_data.split()
        for part in parts:
            if "=" in part:
                key, value = part.split("=", 1)
                try:
                    usage_dict[key] = int(value)
                except ValueError:
                    usage_dict[key] = value
        return {
            "prompt_tokens": usage_dict.get("prompt_tokens", 0),
            "completion_tokens": usage_dict.get("completion_tokens", 0),
            "total_tokens": usage_dict.get("total_tokens", 0)
        }

    else:
        return {
            "prompt_tokens": 0,
            "completion_tokens": 0,
            "total_tokens": 0
        }


def estimate_cost(tokens, cost_per_1k_input=0.001, cost_per_1k_output=0.002):
    """
    Estimate API call cost based on token usage.
    Default prices are approximate. Always check provider pricing.
    """
    input_cost = (tokens["prompt_tokens"] / 1000) * cost_per_1k_input
    output_cost = (tokens["completion_tokens"] / 1000) * cost_per_1k_output
    total_cost = input_cost + output_cost
    return {
        "input_cost_usd": round(input_cost, 6),
        "output_cost_usd": round(output_cost, 6),
        "total_cost_usd": round(total_cost, 6)
    }


# Use the functions
tokens = get_token_usage(response)
print("Token Usage:")
print("  Prompt Tokens     :", tokens["prompt_tokens"])
print("  Completion Tokens :", tokens["completion_tokens"])
print("  Total Tokens      :", tokens["total_tokens"])

cost = estimate_cost(tokens)
print()
print("Estimated Cost:")
print("  Input Cost  : $", cost["input_cost_usd"])
print("  Output Cost : $", cost["output_cost_usd"])
print("  Total Cost  : $", cost["total_cost_usd"])


Token Usage:
  Prompt Tokens     : 20
  Completion Tokens : 2803
  Total Tokens      : 2823

Estimated Cost:
  Input Cost  : $ 2e-05
  Output Cost : $ 0.005606
  Total Cost  : $ 0.005626


### 1.4 Multi-Turn Conversation with SarvamAI

In [8]:
# Real-world example: Customer support conversation
# Business scenario: E-commerce complaint resolution

def sarvam_chat(client, messages, temperature=0.5, max_tokens=3000):
    """Wrapper function for SarvamAI chat completion."""
    response = client.chat.completions(
        model="sarvam-105b",
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content


# Start conversation
conversation = [
    {
        "role": "system",
        "content": (
            "You are a customer support assistant for an Indian e-commerce company. "
            "Answer only based on company policy. "
            "If unsure, ask the customer to escalate to a human agent."
        )
    },
    {
        "role": "user",
        "content": "My order arrived damaged. What should I do?"
    }
]

# First turn
reply_1 = sarvam_chat(client, conversation)
print("Turn 1")
print("Customer: My order arrived damaged. What should I do?")
print("Assistant:", reply_1)

# Append assistant reply to maintain context
conversation.append({"role": "assistant", "content": reply_1})

# Second turn - follow-up question
conversation.append({
    "role": "user",
    "content": "The delivery was 10 days ago. Is it still possible to get a refund?"
})

reply_2 = sarvam_chat(client, conversation)
print()
print("Turn 2")
print("Customer: The delivery was 10 days ago. Is it still possible to get a refund?")
print("Assistant:", reply_2)

# Key concept: Without the full conversation history above,
# the model would not know "delivery" refers to the damaged product from turn 1.
print()
print("Total messages in context:", len(conversation) + 1)


Turn 1
Customer: My order arrived damaged. What should I do?
Assistant: I'm sorry to hear that your order arrived damaged. I can certainly help you with the next steps.

Please follow these instructions:

1.  **Do not use the damaged product.**
2.  **Take clear photos or videos** of the damaged item and its packaging.
3.  **Go to the "Help" or "Contact Us" section** in our app or website.
4.  **Submit a return request** for the damaged item. You will need to upload the photos you took and provide your order ID.

Once we receive your request and verify the damage, we will process either a replacement for the item or a full refund, whichever you prefer. Our team will typically get back to you within 24-48 hours to confirm the next steps.

Is there anything else I can assist you with?

Turn 2
Customer: The delivery was 10 days ago. Is it still possible to get a refund?
Assistant: I understand your concern regarding the timeline.

Our standard return policy allows returns within 7 days of 

---
## Section 2: OpenAI GPT-4o Mini - Complete Parameter Guide

GPT-4o Mini is fast, cost-efficient, and suitable for most business applications.


### 2.1 Basic Setup and Call

In [9]:
from openai import OpenAI

OPENAI_API_KEY = ""

openai_client = OpenAI(api_key=OPENAI_API_KEY)

response = openai_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "What is BM25 and why is it used in search systems?"}
    ],
    temperature=0.5,
    max_tokens=300
)

print(response.choices[0].message.content)


BM25, or Best Matching 25, is a ranking function used in information retrieval systems to evaluate the relevance of documents to a given search query. It is part of the family of probabilistic models and is widely used in search engines and recommendation systems for its effectiveness in ranking documents based on their content and the user's query.

### Key Features of BM25:

1. **Term Frequency (TF):** BM25 takes into account how often a term appears in a document. The idea is that the more frequently a term appears, the more relevant the document is likely to be to that term. However, BM25 applies a saturation effect; after a certain point, adding more occurrences of the term yields diminishing returns.

2. **Inverse Document Frequency (IDF):** BM25 incorporates the concept of IDF, which measures how much information a term provides. A term that appears in many documents is less useful for distinguishing relevant documents than a term that appears in fewer documents. BM25 uses a log

### 2.2 Print Full Response - OpenAI Style

In [10]:
import json

# model_dump() is the recommended way for Pydantic-based OpenAI responses
full_response = response.model_dump()
print(json.dumps(full_response, indent=2))


{
  "id": "chatcmpl-DUqTYAYrlzDo8Wb8mm2GNJprhLCE6",
  "choices": [
    {
      "finish_reason": "length",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "BM25, or Best Matching 25, is a ranking function used in information retrieval systems to evaluate the relevance of documents to a given search query. It is part of the family of probabilistic models and is widely used in search engines and recommendation systems for its effectiveness in ranking documents based on their content and the user's query.\n\n### Key Features of BM25:\n\n1. **Term Frequency (TF):** BM25 takes into account how often a term appears in a document. The idea is that the more frequently a term appears, the more relevant the document is likely to be to that term. However, BM25 applies a saturation effect; after a certain point, adding more occurrences of the term yields diminishing returns.\n\n2. **Inverse Document Frequency (IDF):** BM25 incorporates the concept of IDF, which measu

### 2.3 Token Usage for OpenAI

In [12]:
# OpenAI usage object is always structured (not a string)
# So we can access fields directly

def get_openai_token_usage(response):
    usage = response.usage
    return {
        "prompt_tokens": usage.prompt_tokens,
        "completion_tokens": usage.completion_tokens,
        "total_tokens": usage.total_tokens
    }

tokens = get_openai_token_usage(response)
print("Prompt Tokens     :", tokens["prompt_tokens"])
print("Completion Tokens :", tokens["completion_tokens"])
print("Total Tokens      :", tokens["total_tokens"])

# GPT-4o Mini pricing (approximate as of 2024)
# Input:  $0.150 per 1M tokens
# Output: $0.600 per 1M tokens
cost = estimate_cost(tokens, cost_per_1k_input=0.00015, cost_per_1k_output=0.0006)
print()
print("Estimated Cost (GPT-4o Mini):")
print("  Total: $", cost["total_cost_usd"])


Prompt Tokens     : 20
Completion Tokens : 300
Total Tokens      : 320

Estimated Cost (GPT-4o Mini):
  Total: $ 0.000183


---
## Section 3: Understanding and Experimenting With Parameters

This section covers:
- temperature
- top_p (nucleus sampling)
- top_k
- frequency_penalty
- presence_penalty
- max_tokens

Each parameter is explained with a real-world use case and a before/after comparison.


### 3.1 Temperature

**What it controls:** How random or deterministic the output is.

| Value | Behavior | Use Case |
|-------|----------|----------|
| 0.0 - 0.2 | Very deterministic, near-identical outputs each run | Legal text, policy answers, maths |
| 0.3 - 0.5 | Balanced, slightly varied | Customer support, summaries |
| 0.6 - 0.8 | More natural, conversational | Chatbots, friendly tone |
| 0.9 - 1.0 | Creative, unpredictable | Storytelling, brainstorming |

**Why it matters:** For a customer support system you want temperature 0.2 so the AI does not invent policies. For a creative writing assistant you want 0.9.


In [13]:
# Real-life scenario: E-commerce refund policy question
# Compare how the same question behaves at different temperatures

question = "Can I return a product after 15 days if it is damaged?"

system_prompt = (
    "You are a customer support agent. "
    "Answer based on this policy: "
    "Products can be returned within 7 days of delivery if damaged."
)

temperatures = [0.1, 0.5, 0.9]

for temp in temperatures:
    resp = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ],
        temperature=temp,
        max_tokens=100
    )
    print(f"Temperature = {temp}")
    print(resp.choices[0].message.content)
    print("-" * 60)


Temperature = 0.1
I'm sorry, but according to our policy, products can only be returned within 7 days of delivery if they are damaged. Unfortunately, returns after 15 days cannot be accepted.
------------------------------------------------------------
Temperature = 0.5
Unfortunately, our policy states that products can only be returned within 7 days of delivery if they are damaged. Since it has been 15 days, we are unable to process a return for you. If you have any further questions or need assistance with something else, feel free to ask!
------------------------------------------------------------
Temperature = 0.9
Unfortunately, you cannot return a product after 15 days, even if it is damaged. Our return policy allows for returns within 7 days of delivery for damaged items. If you have any other questions or need assistance, feel free to ask!
------------------------------------------------------------


### 3.2 Top-P (Nucleus Sampling)

**What it controls:** The model only considers tokens whose cumulative probability reaches the top_p threshold.

- top_p = 1.0 means consider all tokens (no filtering)
- top_p = 0.9 means only consider the top 90% probable tokens
- top_p = 0.1 means only consider the most likely tokens (very narrow)

**Rule of thumb:** Change either temperature OR top_p, not both at the same time. Changing both simultaneously makes behavior harder to predict.

**Real-world use:**
- Legal document summarization: top_p = 0.5 (safe, narrow vocabulary)
- Marketing copy: top_p = 0.95 (broad vocabulary, creative)


In [15]:
# Scenario: Financial report summarization
# Low top_p = safe vocabulary, high top_p = varied phrasing

financial_text = (
    "The company reported a 23 percent increase in revenue for Q3, "
    "driven by strong performance in the digital products segment. "
    "Operating expenses rose by 10 percent due to increased hiring."
)

prompt = f"Summarize this financial report in one sentence:{financial_text}"

for top_p in [0.3, 0.7, 1.0]:
    resp = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.5,
        top_p=top_p,
        max_tokens=80
    )
    print(f"top_p = {top_p}")
    print(resp.choices[0].message.content)
    print("-" * 60)


top_p = 0.3
The company experienced a 23% revenue increase in Q3, fueled by robust digital product sales, while operating expenses grew by 10% due to higher hiring costs.
------------------------------------------------------------
top_p = 0.7
The company experienced a 23% revenue increase in Q3, fueled by robust digital product sales, while operating expenses grew by 10% due to higher hiring costs.
------------------------------------------------------------
top_p = 1.0
The company experienced a 23% revenue growth in Q3, fueled by robust digital product sales, while operating expenses increased by 10% due to higher hiring costs.
------------------------------------------------------------


### 3.3 Top-K

**What it controls:** At each step, the model only considers the top K most probable next tokens.

- Lower K = more focused and predictable
- Higher K = more variety

**Note:** OpenAI API does not expose top_k directly. It is available in some providers like OpenRouter, Anthropic, and Cohere.
We will use OpenRouter in Section 4 to demonstrate this.

**Real-world use:**
- top_k = 10 for strict factual Q&A
- top_k = 50 for conversational chatbots
- top_k = 100 for creative text generation


### 3.4 Frequency Penalty

**What it controls:** Reduces repetition of tokens that have already appeared in the output.

- Range: -2.0 to 2.0
- Positive values discourage repeating words that have appeared
- Default: 0 (no penalty)

**Real-world use:**
- Long-form content generation where you do not want the AI to keep repeating the same phrases
- News article drafting


In [16]:
# Scenario: Product description generation
# Without frequency penalty, LLMs often repeat adjectives

product_prompt = "Write a 3-sentence product description for a premium leather wallet."

print("--- Without Frequency Penalty (frequency_penalty = 0.0) ---")
resp_no_penalty = openai_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": product_prompt}],
    temperature=0.7,
    frequency_penalty=0.0,
    max_tokens=120
)
print(resp_no_penalty.choices[0].message.content)

print()
print("--- With Frequency Penalty (frequency_penalty = 1.5) ---")
resp_with_penalty = openai_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": product_prompt}],
    temperature=0.7,
    frequency_penalty=1.5,
    max_tokens=120
)
print(resp_with_penalty.choices[0].message.content)


--- Without Frequency Penalty (frequency_penalty = 0.0) ---
Elevate your everyday essentials with our premium leather wallet, meticulously crafted from the finest full-grain leather for unmatched durability and sophistication. Its sleek design features multiple card slots and a spacious bill compartment, ensuring that your belongings are organized and easily accessible. Experience the perfect blend of style and functionality, making this wallet a timeless accessory for the modern individual.

--- With Frequency Penalty (frequency_penalty = 1.5) ---
Elevate your everyday style with our premium leather wallet, expertly crafted from the finest full-grain leather for a luxurious feel and lasting durability. Featuring a sleek design with ample card slots and a secure bill compartment, this wallet combines functionality with an elegant aesthetic. Whether for yourself or as a thoughtful gift, it's the perfect accessory that exudes sophistication and timeless charm.


### 3.5 Presence Penalty

**What it controls:** Encourages the model to introduce new topics rather than staying on the same topic.

- Range: -2.0 to 2.0
- Unlike frequency_penalty (which scales with count), presence_penalty applies a flat penalty just for appearing
- Useful when you want diverse outputs across multiple ideas

**Real-world use:**
- Brainstorming sessions (want many different ideas)
- Blog post idea generation
- Do NOT use for technical documentation (you want focused output there)


In [17]:
# Scenario: Brainstorming startup ideas vs focused business plan

brainstorm_prompt = "Give me 3 business ideas for a college student in India."

print("--- Low Presence Penalty (tends to stay in one domain) ---")
resp_low = openai_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": brainstorm_prompt}],
    temperature=0.8,
    presence_penalty=0.0,
    max_tokens=150
)
print(resp_low.choices[0].message.content)

print()
print("--- High Presence Penalty (pushes for diverse, distinct ideas) ---")
resp_high = openai_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": brainstorm_prompt}],
    temperature=0.8,
    presence_penalty=1.8,
    max_tokens=150
)
print(resp_high.choices[0].message.content)


--- Low Presence Penalty (tends to stay in one domain) ---
Here are three business ideas that a college student in India could consider:

1. **Online Tutoring Platform**:
   With the rise of online education, college students can leverage their knowledge in specific subjects by offering tutoring services. This can be done through a personalized website or social media platforms. By focusing on specific subjects or competitive exams, students can attract peers looking for help. Additionally, they can create video tutorials or write study guides to sell or share, creating passive income.

2. **Eco-Friendly Products Store**:
   There is a growing awareness of sustainability among consumers in India. A college student could start a business selling eco-friendly products, such as reusable bags, bamboo toothbrushes, or biodegradable utensils. This can be done through an online store

--- High Presence Penalty (pushes for diverse, distinct ideas) ---
Certainly! Here are three business ideas t

### 3.6 Max Tokens

**What it controls:** The maximum number of tokens the model will generate in its response.

- 1 token is roughly 4 characters in English
- Does NOT control the quality of the answer, only the length
- If the answer is cut short mid-sentence, increase max_tokens

**Real-world mapping:**

| Task | Suggested max_tokens |
|------|----------------------|
| Yes/No classification | 5 - 20 |
| Short customer reply | 100 - 200 |
| Policy explanation | 300 - 500 |
| Full article draft | 1000 - 2000 |
| Detailed analysis | 2000 - 4000 |


In [18]:
# Scenario: See how max_tokens cuts off responses

policy_question = (
    "Explain all the steps a customer should follow to "
    "return a damaged product and get a refund."
)

for max_tok in [30, 100, 300]:
    resp = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": policy_question}],
        temperature=0.3,
        max_tokens=max_tok
    )
    print(f"max_tokens = {max_tok}")
    print(resp.choices[0].message.content)
    print(f"(Tokens used: {resp.usage.completion_tokens})")
    print("-" * 60)


max_tokens = 30
Returning a damaged product and obtaining a refund typically involves several steps. While the exact process may vary depending on the retailer's policies, here’s a general
(Tokens used: 30)
------------------------------------------------------------
max_tokens = 100
Returning a damaged product and obtaining a refund typically involves several steps. While the exact process may vary depending on the retailer or manufacturer, here is a general guide that customers can follow:

### Step 1: Review the Return Policy
- **Check the Return Policy**: Before initiating a return, review the retailer's return policy. Look for information on time limits, acceptable conditions for returns, and specific procedures for damaged items.

### Step 2: Gather Necessary Information
- **Order Details**:
(Tokens used: 100)
------------------------------------------------------------
max_tokens = 300
Returning a damaged product and obtaining a refund typically involves several steps. While the

### 3.7 Parameter Summary Table

| Parameter | Range | Low Value Behavior | High Value Behavior | Business Use |
|-----------|-------|-------------------|--------------------|-|
| temperature | 0 - 2 | Deterministic | Creative/random | Low for support, high for creative |
| top_p | 0 - 1 | Narrow vocab | Full vocab | Low for legal, high for marketing |
| top_k | 1 - 100+ | Very focused | Varied | Available via OpenRouter |
| frequency_penalty | -2 - 2 | Allow repetition | Discourage repeats | Use for long-form content |
| presence_penalty | -2 - 2 | Stay on topic | Introduce new topics | Use for brainstorming |
| max_tokens | 1 - 4096+ | Short output | Long output | Match to task type |


---
## Section 4: OpenRouter - Access Multiple Models Through One API

OpenRouter provides a single API endpoint to access:
- Meta LLaMA models
- Mistral
- Google Gemini
- Anthropic Claude
- Cohere
- And many more

This is useful when you want to compare models or use specific models not available directly.

Get your API key at: https://openrouter.ai/keys


### 4.1 OpenRouter Setup - Using OpenAI SDK

In [ ]:
# OpenRouter is compatible with the OpenAI SDK
# Just change the base_url

from openai import OpenAI

OPENROUTER_API_KEY = "your_openrouter_api_key_here"

openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

# Basic call with Meta LLaMA
response = openrouter_client.chat.completions.create(
    model="meta-llama/llama-3.1-8b-instruct:free",  # Free tier model
    messages=[
        {"role": "user", "content": "Explain BM25 in simple terms."}
    ],
    temperature=0.5,
    max_tokens=200
)

print("Model:", response.model)
print()
print(response.choices[0].message.content)


### 4.2 OpenRouter with Extra Headers (Recommended)

In [ ]:
# Adding headers helps OpenRouter track your app usage
# and may unlock higher rate limits

response = openrouter_client.chat.completions.create(
    model="meta-llama/llama-3.1-8b-instruct:free",
    messages=[
        {"role": "user", "content": "What is the refund policy for damaged products?"}
    ],
    temperature=0.3,
    max_tokens=200,
    extra_headers={
        "HTTP-Referer": "https://your-app.com",   # Your app URL
        "X-Title": "Customer Support Assistant",   # App name shown on OpenRouter
    }
)

print(response.choices[0].message.content)


### 4.3 Using Top-K via OpenRouter (Not Available in OpenAI API)

In [ ]:
# Some OpenRouter models support top_k via extra_body
# This is how you pass provider-specific parameters

response = openrouter_client.chat.completions.create(
    model="mistralai/mistral-7b-instruct:free",
    messages=[
        {"role": "system", "content": "You are a strict policy assistant. Do not guess."},
        {"role": "user", "content": "Can I return a product after 30 days?"}
    ],
    temperature=0.2,
    top_p=0.9,
    max_tokens=150,
    extra_body={
        "top_k": 20,          # Only consider top 20 tokens at each step
        "repetition_penalty": 1.1   # Slight penalty for repeated tokens (Mistral style)
    }
)

print("Model:", response.model)
print()
print(response.choices[0].message.content)


### 4.4 Compare Multiple Models on the Same Prompt

In [ ]:
# Business scenario: Choose the best model for a customer support task
# Compare LLaMA vs Mistral on the same complaint

complaint = (
    "I ordered a laptop but received a phone. "
    "I want an immediate replacement and compensation."
)

system_msg = (
    "You are a professional e-commerce support agent. "
    "Respond politely, state the resolution clearly, "
    "and mention how long it will take."
)

models_to_compare = [
    ("meta-llama/llama-3.1-8b-instruct:free", "LLaMA 3.1 8B"),
    ("mistralai/mistral-7b-instruct:free", "Mistral 7B"),
]

results = {}

for model_id, model_name in models_to_compare:
    try:
        resp = openrouter_client.chat.completions.create(
            model=model_id,
            messages=[
                {"role": "system", "content": system_msg},
                {"role": "user", "content": complaint}
            ],
            temperature=0.4,
            max_tokens=200,
        )
        results[model_name] = resp.choices[0].message.content
    except Exception as e:
        results[model_name] = f"Error: {str(e)}"

for model_name, output in results.items():
    print(f"=== {model_name} ===")
    print(output)
    print()


### 4.5 OpenRouter via Direct HTTP Requests

In [ ]:
import requests
import json

# Sometimes you may want raw HTTP control (useful for debugging)

OPENROUTER_API_KEY = "your_openrouter_api_key_here"

url = "https://openrouter.ai/api/v1/chat/completions"

headers = {
    "Authorization": f"Bearer {OPENROUTER_API_KEY}",
    "Content-Type": "application/json",
    "HTTP-Referer": "https://your-app.com",
    "X-Title": "LLM Notebook"
}

payload = {
    "model": "meta-llama/llama-3.1-8b-instruct:free",
    "messages": [
        {"role": "user", "content": "Explain what prompt engineering means in one paragraph."}
    ],
    "temperature": 0.5,
    "max_tokens": 200,
    "top_p": 0.9
}

response_raw = requests.post(url, headers=headers, json=payload)

if response_raw.status_code == 200:
    data = response_raw.json()
    print(data["choices"][0]["message"]["content"])
    print()
    print("Token Usage:")
    print(json.dumps(data.get("usage", {}), indent=2))
else:
    print("Error:", response_raw.status_code)
    print(response_raw.text)


---
## Section 5: Chat Models vs Completion Models - With Real Examples

Understanding this difference matters when reading documentation, choosing APIs, and explaining your architecture.


### 5.1 Conceptual Difference

**Completion models** (legacy):
- Input: a plain string prompt
- Output: continuation of that string
- No concept of roles or conversation structure
- Examples: text-davinci-003, davinci, GPT-3

**Chat models** (current standard):
- Input: a structured list of messages with roles
- Output: the next assistant message
- Built for multi-turn conversation
- Examples: GPT-4o, GPT-4o Mini, Claude, Gemini, LLaMA 3

**Why chat models replaced completion models:**
Chat models are better at following instructions, maintaining context, and respecting role-based boundaries (system / user / assistant).


In [ ]:
# Completion-style thinking vs Chat-style thinking
# Even though we use the chat endpoint, understanding the mental model matters

# COMPLETION STYLE (old way of thinking)
# You would concatenate context manually:

old_style_prompt = (
    "Policy: Products can be returned within 7 days of delivery.\n\n"
    "Customer: I received a damaged product 5 days ago. Can I return it?\n"
    "Support Agent:"
)
# The model would just complete this text.

# CHAT STYLE (current standard)
# You define clear roles and structure:

chat_style_messages = [
    {
        "role": "system",
        "content": "Policy: Products can be returned within 7 days of delivery."
    },
    {
        "role": "user",
        "content": "I received a damaged product 5 days ago. Can I return it?"
    }
]

# The model predicts the assistant reply, not just text completion.

# Let us see both approaches produce similar output with chat endpoint

resp_old_style = openai_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": old_style_prompt}],
    temperature=0.3,
    max_tokens=100
)

resp_chat_style = openai_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=chat_style_messages,
    temperature=0.3,
    max_tokens=100
)

print("Old completion-style prompt result:")
print(resp_old_style.choices[0].message.content)
print()
print("Structured chat-style result:")
print(resp_chat_style.choices[0].message.content)


---
## Section 6: BM25 Retrieval + LLM Integration

**What is BM25?**

BM25 (Best Match 25) is a ranking function used in search engines to rank documents by relevance to a query.
It improves on TF-IDF by:
- Normalizing for document length
- Applying a saturation function to term frequency

**Why use BM25 instead of vector search?**
- No external API needed
- No embedding model required
- Fast on small to medium datasets (up to ~10k documents)
- Transparent: you can see exactly why a document scored high
- Fits the business constraint: no vector DB allowed


### 6.1 Load and Prepare the Dataset

In [21]:
import json

# This is a sample of the e-commerce support dataset
# In a real project, load from data.json

DATASET = [
    {
        "id": 1,
        "trouble": "Product arrived damaged on delivery",
        "category": "Returns",
        "solution": "Request return within 7 days",
        "alternate_solution": "Offer partial refund",
        "company_response": (
            "We are sorry your product arrived damaged. "
            "Please initiate a return within 7 days for a full refund."
        )
    },
    {
        "id": 2,
        "trouble": "Order delayed beyond promised date",
        "category": "Delivery",
        "solution": "Check tracking and update delivery timeline",
        "alternate_solution": "Provide apology coupon",
        "company_response": (
            "We regret the delay. Your order will arrive soon and we appreciate your patience."
        )
    },
    {
        "id": 3,
        "trouble": "Refund not received after return",
        "category": "Payments",
        "solution": "Verify refund status and reprocess",
        "alternate_solution": "Escalate to finance team",
        "company_response": (
            "Your refund is being processed. Please allow 5 to 7 business days for it to reflect."
        )
    },
    {
        "id": 4,
        "trouble": "Received wrong item in order",
        "category": "Returns",
        "solution": "Arrange pickup and send correct product",
        "alternate_solution": "Issue full refund if replacement unavailable",
        "company_response": (
            "We apologize for the incorrect item. A replacement has been initiated."
        )
    },
    {
        "id": 5,
        "trouble": "Unable to login to account",
        "category": "Account",
        "solution": "Reset password using email link",
        "alternate_solution": "Provide manual reset via support",
        "company_response": (
            "Please reset your password using the link provided. "
            "Contact support if the issue persists."
        )
    },
    {
        "id": 6,
        "trouble": "Payment failed but money deducted",
        "category": "Payments",
        "solution": "Check transaction and initiate refund",
        "alternate_solution": "Advise contacting bank",
        "company_response": (
            "We have identified the payment issue. A refund will be processed within 5 business days."
        )
    },
    {
        "id": 7,
        "trouble": "App crashes during checkout",
        "category": "Technical",
        "solution": "Clear cache and retry",
        "alternate_solution": "Update app to latest version",
        "company_response": (
            "Please clear your app cache or update to the latest version and try again."
        )
    },
    {
        "id": 8,
        "trouble": "Coupon not working at checkout",
        "category": "Offers",
        "solution": "Check eligibility and conditions",
        "alternate_solution": "Provide alternate coupon",
        "company_response": (
            "The coupon may not be applicable to your current order. "
            "We have provided an alternate offer for your convenience."
        )
    },
    {
        "id": 9,
        "trouble": "Item missing from package",
        "category": "Returns",
        "solution": "Verify and process refund",
        "alternate_solution": "Ship missing item separately",
        "company_response": (
            "We are sorry an item was missing from your order. We will process your refund shortly."
        )
    },
    {
        "id": 10,
        "trouble": "Delivery agent did not arrive",
        "category": "Delivery",
        "solution": "Reschedule delivery",
        "alternate_solution": "Escalate to courier partner",
        "company_response": (
            "We are coordinating with the delivery team to resolve this and reschedule your delivery."
        )
    },
    {
        "id": 11,
        "trouble": "Discount not applied correctly",
        "category": "Offers",
        "solution": "Validate offer rules and apply manually",
        "alternate_solution": "Provide a replacement coupon",
        "company_response": (
            "The discount will be applied to your order shortly after verification."
        )
    },
    {
        "id": 12,
        "trouble": "Order stuck in processing stage",
        "category": "Orders",
        "solution": "Check backend status and update customer",
        "alternate_solution": "Escalate to operations team",
        "company_response": (
            "We are resolving the processing delay and will update you within 24 hours."
        )
    },
    {
        "id": 13,
        "trouble": "Account locked after multiple login attempts",
        "category": "Account",
        "solution": "Verify identity and unlock account",
        "alternate_solution": "Wait 24 hours before retry",
        "company_response": (
            "Your account has been temporarily locked for security. "
            "Please verify your identity to unlock it."
        )
    },
    {
        "id": 14,
        "trouble": "Tracking not updating after dispatch",
        "category": "Delivery",
        "solution": "Inform customer of tracking delay",
        "alternate_solution": "Contact courier service directly",
        "company_response": (
            "Tracking updates may be delayed by up to 24 hours. "
            "We are verifying the status with our courier."
        )
    },
    {
        "id": 15,
        "trouble": "Duplicate payment charged",
        "category": "Payments",
        "solution": "Refund the duplicate transaction",
        "alternate_solution": "Escalate to finance team",
        "company_response": (
            "We have identified the duplicate charge. "
            "A full refund for the extra payment will be processed within 3 business days."
        )
    },
]

print(f"Dataset loaded: {len(DATASET)} records")
print()
print("Sample record:")
print(json.dumps(DATASET[0], indent=2))


Dataset loaded: 15 records

Sample record:
{
  "id": 1,
  "trouble": "Product arrived damaged on delivery",
  "category": "Returns",
  "solution": "Request return within 7 days",
  "alternate_solution": "Offer partial refund",
  "company_response": "We are sorry your product arrived damaged. Please initiate a return within 7 days for a full refund."
}


### 6.2 Build the BM25 Index

In [22]:
from rank_bm25 import BM25Okapi
import re

def preprocess_text(text):
    """
    Clean and tokenize text for BM25 indexing.
    Lowercases, removes punctuation, and splits into tokens.
    """
    text = text.lower()
    text = re.sub(r'[^a-z0-9 ]', ' ', text)
    tokens = text.split()
    return tokens


def build_bm25_index(dataset):
    """
    Build a BM25 index from the dataset.
    Uses: trouble + category + solution + alternate_solution as the document text.
    Returns the BM25 object and the original documents.
    """
    corpus = []
    for record in dataset:
        combined_text = (
            record["trouble"] + " " +
            record["category"] + " " +
            record["solution"] + " " +
            record["alternate_solution"]
        )
        tokens = preprocess_text(combined_text)
        corpus.append(tokens)

    bm25 = BM25Okapi(corpus)
    return bm25, dataset


# Build the index
bm25_index, documents = build_bm25_index(DATASET)
print("BM25 index built successfully.")
print(f"Indexed {len(documents)} documents.")


BM25 index built successfully.
Indexed 15 documents.


### 6.3 Retrieve Top Documents Using BM25

In [23]:
def retrieve_top_k(query, bm25, documents, top_k=3):
    """
    Given a query string, retrieve the top-k most relevant documents.
    Returns a list of (score, document) tuples sorted by relevance.
    """
    query_tokens = preprocess_text(query)
    scores = bm25.get_scores(query_tokens)

    # Pair each document with its score
    scored_docs = list(zip(scores, documents))

    # Sort by score descending
    scored_docs.sort(key=lambda x: x[0], reverse=True)

    return scored_docs[:top_k]


# Test the retrieval
test_query = "I paid but my order did not go through and money was taken"

results = retrieve_top_k(test_query, bm25_index, documents, top_k=3)

print(f"Query: {test_query}")
print()
print("Top 3 Retrieved Documents:")
print("-" * 60)

for rank, (score, doc) in enumerate(results, 1):
    print(f"Rank {rank} | Score: {score:.4f}")
    print(f"  Trouble  : {doc['trouble']}")
    print(f"  Category : {doc['category']}")
    print(f"  Solution : {doc['solution']}")
    print(f"  Response : {doc['company_response']}")
    print()


Query: I paid but my order did not go through and money was taken

Top 3 Retrieved Documents:
------------------------------------------------------------
Rank 1 | Score: 5.1386
  Trouble  : Payment failed but money deducted
  Category : Payments
  Solution : Check transaction and initiate refund
  Response : We have identified the payment issue. A refund will be processed within 5 business days.

Rank 2 | Score: 3.1693
  Trouble  : Delivery agent did not arrive
  Category : Delivery
  Solution : Reschedule delivery
  Response : We are coordinating with the delivery team to resolve this and reschedule your delivery.

Rank 3 | Score: 1.7575
  Trouble  : Order delayed beyond promised date
  Category : Delivery
  Solution : Check tracking and update delivery timeline
  Response : We regret the delay. Your order will arrive soon and we appreciate your patience.



### 6.4 Format Retrieved Docs as Context for LLM

In [24]:
def format_context(retrieved_docs):
    """
    Convert retrieved BM25 results into a readable context string
    that can be injected into an LLM prompt.
    """
    context_parts = []
    for rank, (score, doc) in enumerate(retrieved_docs, 1):
        part = (
            f"[Document {rank} | Category: {doc['category']} | BM25 Score: {score:.3f}]\n"
            f"Issue: {doc['trouble']}\n"
            f"Solution: {doc['solution']}\n"
            f"Alternate: {doc['alternate_solution']}\n"
            f"Standard Response: {doc['company_response']}"
        )
        context_parts.append(part)
    return "\n\n".join(context_parts)


# Test context formatting
context = format_context(results)
print("Formatted Context for LLM:")
print("-" * 60)
print(context)


Formatted Context for LLM:
------------------------------------------------------------
[Document 1 | Category: Payments | BM25 Score: 5.139]
Issue: Payment failed but money deducted
Solution: Check transaction and initiate refund
Alternate: Advise contacting bank
Standard Response: We have identified the payment issue. A refund will be processed within 5 business days.

[Document 2 | Category: Delivery | BM25 Score: 3.169]
Issue: Delivery agent did not arrive
Solution: Reschedule delivery
Alternate: Escalate to courier partner
Standard Response: We are coordinating with the delivery team to resolve this and reschedule your delivery.

[Document 3 | Category: Delivery | BM25 Score: 1.758]
Issue: Order delayed beyond promised date
Solution: Check tracking and update delivery timeline
Alternate: Provide apology coupon
Standard Response: We regret the delay. Your order will arrive soon and we appreciate your patience.


---
## Section 7: Prompt Engineering Techniques

This section implements the key prompt engineering techniques required for the assignment.


### 7.1 Technique 1: Context Injection (Grounded Answering)

Inject retrieved documents into the prompt so the LLM answers ONLY from known information.
This prevents hallucination.


In [25]:
def build_strict_policy_prompt(context, customer_query):
    """
    Strict mode: LLM must answer ONLY from the provided context.
    Use temperature 0.1 - 0.3.
    """
    return f"""You are a professional customer support assistant for an e-commerce company.

IMPORTANT RULES:
- Answer ONLY using the policy information in the Context below.
- Do NOT make up policies, timelines, or amounts.
- If the context does not cover the issue, respond with:
  "This issue requires escalation to a human support agent."
- Be concise and direct.

Context:
{context}

Customer Query:
{customer_query}

Response:"""


# Test
query = "I paid for an order but the payment was deducted and the order failed. What happens now?"
retrieved = retrieve_top_k(query, bm25_index, documents, top_k=3)
context = format_context(retrieved)

prompt = build_strict_policy_prompt(context, query)

resp = openai_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}],
    temperature=0.2,
    max_tokens=200
)

print("Strict Policy Mode Response:")
print(resp.choices[0].message.content)
print()
print("Token Usage:", get_openai_token_usage(resp))


Strict Policy Mode Response:
We have identified the payment issue. A refund will be processed within 5 business days.

Token Usage: {'prompt_tokens': 293, 'completion_tokens': 18, 'total_tokens': 311}


### 7.2 Technique 2: Role Prompting

Give the LLM a specific persona to control tone and behavior.


In [26]:
def build_friendly_tone_prompt(context, customer_query):
    """
    Friendly mode: Same policy, but warmer tone.
    Use temperature 0.6 - 0.7.
    """
    return f"""You are a warm, empathetic customer support agent for an e-commerce company.

Your personality:
- Polite and understanding
- Use phrases like "I understand how frustrating this must be"
- Still accurate and policy-based, but human in tone
- End with a reassurance

Company Policy Context:
{context}

Customer Query:
{customer_query}

Your Response:"""


resp_friendly = openai_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": build_friendly_tone_prompt(context, query)}],
    temperature=0.7,
    max_tokens=200
)

print("Friendly Tone Mode Response:")
print(resp_friendly.choices[0].message.content)


Friendly Tone Mode Response:
I understand how frustrating this must be, especially when you've already made the payment. I want to assure you that we will take care of this. We have identified the payment issue, and a refund will be processed within 5 business days. 

In the meantime, if you would like, I can help you check the transaction details or advise you to contact your bank for further assistance. Please know that we are here to support you throughout this process. Thank you for your patience!


### 7.3 Technique 3: Chain-of-Thought (Hidden Reasoning)

Force the model to reason step by step internally before giving the final answer.
Useful for complex queries with multiple conditions.


In [27]:
def build_chain_of_thought_prompt(context, customer_query):
    """
    CoT mode: Model reasons through the problem step by step.
    The reasoning helps produce a more accurate final answer.
    """
    return f"""You are a senior customer support analyst.

Given the context and query below, reason through the following steps:

Step 1: Identify what category of issue this is.
Step 2: Find the relevant policy from the context.
Step 3: Check if any conditions or exceptions apply.
Step 4: Decide the correct resolution.
Step 5: Write the final customer response.

Output format:
[Reasoning]
(your step-by-step analysis here)

[Final Response to Customer]
(the actual reply the customer should receive)

Context:
{context}

Customer Query:
{customer_query}"""


complex_query = (
    "My product arrived damaged 10 days ago. "
    "I tried to report it earlier but the app was crashing. "
    "Can I still get a return or refund given the technical issue?"
)

retrieved_complex = retrieve_top_k(complex_query, bm25_index, documents, top_k=3)
context_complex = format_context(retrieved_complex)

resp_cot = openai_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": build_chain_of_thought_prompt(context_complex, complex_query)}],
    temperature=0.4,
    max_tokens=400
)

print("Chain-of-Thought Response:")
print(resp_cot.choices[0].message.content)


Chain-of-Thought Response:
[Reasoning]
Step 1: The issue falls under the category of "Returns" since the customer is reporting that their product arrived damaged and is seeking a return or refund.

Step 2: The relevant policy from Document 1 states that a return must be requested within 7 days for a full refund. 

Step 3: The customer mentions that they experienced a technical issue (app crashing) which prevented them from reporting the damage sooner. However, the policy does not specify any exceptions for technical issues affecting the return timeframe.

Step 4: Since the customer is reporting the issue after the 7-day return window, the standard policy would suggest that they are not eligible for a full refund. However, considering the circumstances of the app crashing, it may be reasonable to offer a partial refund as an alternative solution.

Step 5: The final response should acknowledge the customer's situation, explain the return policy, and offer a solution that aligns with the 

### 7.4 Technique 4: Structured Output Prompt

Force the model to respond in a specific format (important for frontend integration).


In [28]:
def build_structured_output_prompt(context, customer_query):
    """
    Structured mode: Forces JSON-style output.
    Easy to parse in a React frontend.
    """
    return f"""You are a customer support assistant.

Using the context provided, respond to the customer query in the following EXACT format:

Answer: <your policy-based answer here>
Sources: <list the Document numbers from context that you used, e.g., Document 1, Document 3>
Escalate: <Yes or No - should this be sent to a human agent>
Confidence: <High, Medium, or Low>

Rules:
- Use only information from the context
- If no relevant context found, set Escalate to Yes
- Be concise in Answer (2-3 sentences maximum)

Context:
{context}

Customer Query:
{customer_query}"""


resp_structured = openai_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": build_structured_output_prompt(context, query)}],
    temperature=0.2,
    max_tokens=200
)

print("Structured Output Response:")
print(resp_structured.choices[0].message.content)


Structured Output Response:
Answer: We have identified the payment issue. A refund will be processed within 5 business days. Please check your transaction details and consider contacting your bank for further assistance.  
Sources: Document 1  
Escalate: No  
Confidence: High


### 7.5 Fallback Prompt: No Relevant Context Found

When BM25 scores are too low, no context is available. Handle this gracefully.


In [29]:
def is_low_relevance(retrieved_docs, threshold=0.5):
    """
    Check if the best BM25 score is below a threshold.
    If true, the retrieved context is unreliable.
    """
    if not retrieved_docs:
        return True
    best_score = retrieved_docs[0][0]
    return best_score < threshold


def build_fallback_prompt(customer_query):
    return f"""You are a customer support assistant.

No relevant policy information was found for the following query.

Customer Query:
{customer_query}

Respond with:
"We were unable to find a policy that covers your specific issue.
Your request has been escalated to a senior support agent who will contact you within 24 hours.""""


# Test with an unusual query
unusual_query = "I want to use my refund to buy cryptocurrency through your platform"

retrieved_unusual = retrieve_top_k(unusual_query, bm25_index, documents, top_k=3)

print("BM25 Scores for unusual query:")
for score, doc in retrieved_unusual:
    print(f"  Score {score:.4f}: {doc['trouble']}")

print()

if is_low_relevance(retrieved_unusual, threshold=1.0):
    print("Low relevance detected. Using fallback prompt.")
    fallback_resp = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": build_fallback_prompt(unusual_query)}],
        temperature=0.2,
        max_tokens=100
    )
    print(fallback_resp.choices[0].message.content)
else:
    print("Sufficient context found. Proceeding with normal response.")


SyntaxError: unterminated string literal (detected at line 22) (3782661614.py, line 22)

---
## Section 8: Full End-to-End Pipeline

This brings everything together into a single reusable function.


In [ ]:
def customer_support_pipeline(
    customer_query,
    mode="strict",
    top_k=3,
    bm25_threshold=0.5
):
    """
    Complete pipeline:
    1. Retrieve relevant docs using BM25
    2. Check if results are relevant enough
    3. Select appropriate prompt based on mode
    4. Call LLM with correct parameters
    5. Return structured result

    Modes: 'strict', 'friendly', 'chain_of_thought', 'structured'
    """

    # Step 1: BM25 Retrieval
    retrieved = retrieve_top_k(customer_query, bm25_index, documents, top_k=top_k)

    # Step 2: Check relevance
    if is_low_relevance(retrieved, threshold=bm25_threshold):
        prompt = build_fallback_prompt(customer_query)
        temperature = 0.1
        max_tokens = 100
        mode_used = "fallback"
    else:
        context = format_context(retrieved)
        mode_used = mode

        if mode == "strict":
            prompt = build_strict_policy_prompt(context, customer_query)
            temperature = 0.2
            max_tokens = 200
        elif mode == "friendly":
            prompt = build_friendly_tone_prompt(context, customer_query)
            temperature = 0.7
            max_tokens = 250
        elif mode == "chain_of_thought":
            prompt = build_chain_of_thought_prompt(context, customer_query)
            temperature = 0.4
            max_tokens = 400
        elif mode == "structured":
            prompt = build_structured_output_prompt(context, customer_query)
            temperature = 0.2
            max_tokens = 200
        else:
            prompt = build_strict_policy_prompt(context, customer_query)
            temperature = 0.3
            max_tokens = 200

    # Step 3: LLM Call
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=max_tokens
    )

    # Step 4: Collect results
    result = {
        "query": customer_query,
        "mode": mode_used,
        "temperature": temperature,
        "response": response.choices[0].message.content,
        "retrieved_docs": [
            {
                "rank": i + 1,
                "score": float(score),
                "trouble": doc["trouble"],
                "category": doc["category"]
            }
            for i, (score, doc) in enumerate(retrieved)
        ],
        "token_usage": get_openai_token_usage(response)
    }

    return result


# Test the full pipeline
test_queries = [
    ("My product arrived damaged and I want a refund", "strict"),
    ("The delivery guy never showed up and I am very frustrated", "friendly"),
    ("I paid but order failed and it has been 8 days with no refund, also app crashed when I tried to report", "chain_of_thought"),
    ("I never received my refund after returning the wrong item", "structured"),
]

for query, mode in test_queries:
    print(f"Query : {query}")
    print(f"Mode  : {mode}")
    result = customer_support_pipeline(query, mode=mode)
    print(f"Temp  : {result['temperature']}")
    print("Response:")
    print(result["response"])
    print()
    print("Retrieved Documents:")
    for doc in result["retrieved_docs"]:
        print(f"  Rank {doc['rank']} | Score {doc['score']:.3f} | {doc['trouble']} [{doc['category']}]")
    print()
    print("Tokens:", result["token_usage"])
    print("=" * 70)
    print()


---
## Section 9: Parameter Experimentation - Business Scenarios Matrix

This section runs a matrix experiment showing how different parameter combinations
produce different outputs for the same business problem.
This is useful for choosing the right settings per use case.


In [ ]:
# Business scenario: Customer filed a complaint about a late delivery
# We test different parameter combinations

test_complaint = "My order is 5 days late and I have an important event tomorrow."

PARAMETER_SCENARIOS = [
    {
        "name": "Strict Policy (SLA Compliance Use)",
        "description": "Used for compliance-critical responses",
        "temperature": 0.1,
        "top_p": 0.5,
        "max_tokens": 100,
        "frequency_penalty": 0.0,
        "presence_penalty": 0.0
    },
    {
        "name": "Balanced Support (Standard Agent)",
        "description": "Default setting for most support tickets",
        "temperature": 0.5,
        "top_p": 0.9,
        "max_tokens": 180,
        "frequency_penalty": 0.3,
        "presence_penalty": 0.1
    },
    {
        "name": "Empathetic Tone (VIP Customer)",
        "description": "For high-value customers or escalated cases",
        "temperature": 0.75,
        "top_p": 1.0,
        "max_tokens": 220,
        "frequency_penalty": 0.5,
        "presence_penalty": 0.3
    },
    {
        "name": "Creative Compensation (Retention Team)",
        "description": "When agents need to retain an unhappy customer",
        "temperature": 0.9,
        "top_p": 1.0,
        "max_tokens": 250,
        "frequency_penalty": 0.8,
        "presence_penalty": 0.5
    },
]

system_message = (
    "You are a customer support agent. "
    "Policy: Deliveries delayed more than 3 days receive a 10 percent discount coupon. "
    "Respond to the customer professionally."
)

print(f"Customer Complaint: {test_complaint}")
print("=" * 70)

for scenario in PARAMETER_SCENARIOS:
    resp = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": test_complaint}
        ],
        temperature=scenario["temperature"],
        top_p=scenario["top_p"],
        max_tokens=scenario["max_tokens"],
        frequency_penalty=scenario["frequency_penalty"],
        presence_penalty=scenario["presence_penalty"]
    )

    print(f"Scenario: {scenario['name']}")
    print(f"Use Case: {scenario['description']}")
    print(f"Parameters: temp={scenario['temperature']} | top_p={scenario['top_p']} | max_tokens={scenario['max_tokens']}")
    print(f"            freq_penalty={scenario['frequency_penalty']} | pres_penalty={scenario['presence_penalty']}")
    print()
    print("Response:")
    print(resp.choices[0].message.content)
    print()
    print("-" * 70)
    print()


---
## Section 10: Production-Ready Patterns

These are patterns used in real systems, not just demos.


### 10.1 Error Handling and Retry Logic

In [30]:
import time

def safe_llm_call(client, messages, model="gpt-4o-mini", temperature=0.5, max_tokens=200, retries=3):
    """
    Production-ready LLM call with error handling and exponential backoff.
    """
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=messages,
                temperature=temperature,
                max_tokens=max_tokens
            )
            return response

        except Exception as e:
            error_str = str(e).lower()

            if "rate_limit" in error_str or "429" in error_str:
                wait_time = 2 ** attempt  # 1s, 2s, 4s
                print(f"Rate limit hit. Waiting {wait_time}s before retry {attempt + 1}/{retries}")
                time.sleep(wait_time)

            elif "invalid_api_key" in error_str or "401" in error_str:
                print("Authentication error. Check your API key.")
                return None

            elif "context_length" in error_str or "token" in error_str:
                print("Context too long. Reducing prompt size.")
                # In production: truncate oldest messages
                if len(messages) > 2:
                    messages = [messages[0]] + messages[-2:]
                    continue
                return None

            else:
                print(f"Unexpected error on attempt {attempt + 1}: {e}")
                if attempt == retries - 1:
                    return None

    return None


# Test with actual call
messages = [{"role": "user", "content": "What is prompt engineering?"}]
result = safe_llm_call(openai_client, messages, max_tokens=150)

if result:
    print(result.choices[0].message.content)
else:
    print("Call failed after all retries.")


Prompt engineering is the process of designing and refining prompts to effectively communicate with artificial intelligence models, particularly language models like GPT-3 and GPT-4. The goal is to elicit the most accurate, relevant, and useful responses from the model by carefully crafting the input it receives.

Key aspects of prompt engineering include:

1. **Clarity**: Ensuring the prompt is clear and unambiguous, so the model understands what is being asked.

2. **Specificity**: Providing specific instructions or context to guide the model towards the desired output. This can include details about the format, tone, or style of the response.

3. **Context**: Including relevant background information that can help the model generate a more informed response.

4.


### 10.3 Conversation Memory Manager

In [32]:
def manage_conversation_window(messages, max_tokens_budget=2000, avg_tokens_per_message=50):
    """
    Prevent context window overflow in long conversations.
    Keeps the system message and trims oldest user/assistant turns.

    In production: use tiktoken to count tokens precisely.
    Here we use a rough estimate.
    """
    # Always keep system message
    system_messages = [m for m in messages if m["role"] == "system"]
    non_system = [m for m in messages if m["role"] != "system"]

    # Estimate current token usage
    estimated_tokens = len(messages) * avg_tokens_per_message

    while estimated_tokens > max_tokens_budget and len(non_system) > 2:
        # Remove the oldest user+assistant pair
        non_system = non_system[2:]
        estimated_tokens = (len(system_messages) + len(non_system)) * avg_tokens_per_message

    managed = system_messages + non_system

    print(f"Original messages: {len(messages)} | After trimming: {len(managed)}")
    return managed


# Simulate a long conversation
long_conversation = [
    {"role": "system", "content": "You are a customer support agent."},
    {"role": "user", "content": "My order is late."},
    {"role": "assistant", "content": "We apologize, it will arrive soon."},
    {"role": "user", "content": "How late exactly?"},
    {"role": "assistant", "content": "Tracking shows 2 more days."},
    {"role": "user", "content": "That is too long. I want a refund."},
    {"role": "assistant", "content": "Please initiate a return from your account."},
    {"role": "user", "content": "I already did but nothing happened."},
    {"role": "assistant", "content": "We will escalate this to our finance team."},
    {"role": "user", "content": "When will the refund appear?"},
]

trimmed = manage_conversation_window(long_conversation, max_tokens_budget=300)

print()
print("Trimmed Conversation:")
for msg in trimmed:
    print(f"  [{msg['role']}]: {msg['content']}")


Original messages: 10 | After trimming: 6

Trimmed Conversation:
  [system]: You are a customer support agent.
  [user]: That is too long. I want a refund.
  [assistant]: Please initiate a return from your account.
  [user]: I already did but nothing happened.
  [assistant]: We will escalate this to our finance team.
  [user]: When will the refund appear?


---
## Section 11: Summary and Key Takeaways

### What Was Covered

**SarvamAI API**
- Basic calls, response parsing, token usage calculation
- Multi-turn conversations with context maintenance

**OpenAI GPT-4o Mini**
- Full parameter control: temperature, top_p, frequency_penalty, presence_penalty, max_tokens
- Real-world scenario comparison across parameter settings

**OpenRouter**
- Accessing multiple models through one API (LLaMA, Mistral, etc.)
- Using top_k via extra_body for models that support it
- Direct HTTP requests for debugging

**BM25 Retrieval**
- Index building from structured dataset
- Query preprocessing and tokenization
- Top-k retrieval with scoring

**Prompt Engineering Techniques Implemented**
1. Context injection for grounded answering
2. Role prompting for tone control
3. Chain-of-thought for complex multi-condition queries
4. Structured output for frontend integration
5. Fallback handling when no relevant context is found

**Production Patterns**
- Error handling with exponential backoff
- Session logging for cost tracking
- Conversation window management

### Parameter Decision Guide

When building your own application, choose parameters based on the task:

- Legal or compliance content: temperature 0.1, top_p 0.5
- Standard customer support: temperature 0.3-0.5, top_p 0.9
- Friendly conversational tone: temperature 0.6-0.7, top_p 1.0
- Creative content generation: temperature 0.8-0.9, presence_penalty 1.0+
- Long-form content with no repetition: frequency_penalty 0.8-1.5

### Why Context Maintenance Requires Full Message History

The model has no memory between API calls.
Every call is stateless.
Context is preserved only because YOU send the full conversation history in messages.
The assistant role does not create memory. It represents what the model previously said so it can respond coherently.
